# 50. Multistart fits with `fit_multistart`

**Objectives:**

- Build a small model and run `FitSession.fit_multistart` from several randomized starts.
- Inspect `MultiStartResult.valid_results` and `MultiStartResult.best`.
- Check how many independent starts converge to the same minimum.

Run the cells in order in a fresh kernel. Masses are in GeV, invariants in GeV^2, and daughter
indices start at zero.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede any numerical work: amplitudes use complex128.

import numpy as np

from dalitzplotfitter import (
    DecayChannel, DecayModel, FitSession, NonResonant,
    Parameter, RealImag, Resonance, generate_toy,
)

## 1. Model and toy data

A rho(770) with a fixed coefficient plus a non-resonant term with a floatable Cartesian
coefficient `NR.x`/`NR.y`. The toy is generated at the true values; `fit_multistart` below is
given only randomized starting points, never the truth.

In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
x = Parameter.coefficient("NR.x", 0.55, owner="NR", bounds=(-2, 2), step=0.02)
y = Parameter.coefficient("NR.y", 0.30, owner="NR", bounds=(-2, 2), step=0.02)
components = [
    Resonance("rho", (0, 1), RealImag(1, 0), mass=0.7753, width=0.1491, spin=1),
    NonResonant(RealImag(x, y), name="NR"),
]
model = DecayModel(
    channel, components, normalization_method="square-dalitz",
    normalization_resolution=60, normalization_pair=(0, 1),
)
truth = {p.name: p.value for p in model.parameters}

data = generate_toy(
    model, 1500, parameters=truth, seed=2026,
    method="inverse-transform", inverse_resolution=256, include_momenta=False,
)
session = FitSession(model, data)
print(f"Generated {data.size} unweighted events; truth = {truth}")

Generated 1500 unweighted events; truth = {'NR.x': 0.55, 'NR.y': 0.3}


## 2. `fit_multistart`

`FitSession.fit_multistart(n_starts=...)` draws `n_starts` random initial values for every free
parameter (`Minimizer.random_start`/`_draw_parameter`), fits each independently without HESSE,
then re-runs the best valid, finite minimum with HESSE enabled. It returns a `MultiStartResult`
with `.results` (one entry per start, in order), `.starts` (the initial values used) and `.best`
(the refined best minimum). Keep `n_starts` small here so the lesson stays fast.

In [3]:
multistart = session.fit_multistart(n_starts=6, seed=7, strategy=1)
print(f"starts requested       : 6")
print(f"valid, finite results   : {len(multistart.valid_results)}")
print(f"best NLL                : {float(multistart.best.fval):.6f}")
for name in multistart.best.parameters:
    print(f"  {name}: best={float(multistart.best.values[name]):.4f}  truth={truth[name]:.4f}")

starts requested       : 6
valid, finite results   : 6
best NLL                : 1854.908813
  NR.x: best=0.6057  truth=0.5500
  NR.y: best=0.3122  truth=0.3000


## 3. Do the starts agree on a minimum?

Compare every valid start's final NLL to the best one. Values matching within a small tolerance
converged to the same minimum; a Cartesian coefficient pair on a single non-resonant term has
essentially one minimum here, so we expect most (or all) valid starts to agree.

In [4]:
best_fval = float(multistart.best.fval)
agreeing = sum(
    1 for result in multistart.valid_results
    if abs(float(result.fval) - best_fval) < 1e-3
)
print(f"valid starts                          : {len(multistart.valid_results)}")
print(f"starts agreeing with the best minimum : {agreeing}")
for index, result in enumerate(multistart.results, start=1):
    status = "valid" if bool(result.valid) else "invalid"
    print(f"start {index}: {status:8s} NLL={float(result.fval):.6f}")

valid starts                          : 6
starts agreeing with the best minimum : 5
start 1: valid    NLL=1854.908813
start 2: valid    NLL=1854.908813
start 3: valid    NLL=1854.908813
start 4: valid    NLL=2649.931416
start 5: valid    NLL=1854.908813
start 6: valid    NLL=1854.908813


## Summary and exercises

1. Raise `n_starts` and see whether more starts still agree on the same minimum.
2. Pass `include_default=True` to also fit from the model's own current values.
3. Compare `multistart.best` against a plain `session.fit(...)` call from a single reasonable
   starting point -- they should agree when the likelihood has one dominant minimum.

Return to the [course guide](TUTORIALS.md).